# RADAR Inference 


## 1) Paths

All paths below follow your existing project layout. You only need to change `RADAR_REPO_DIR` if you'd rather clone the repo somewhere else.


In [1]:
from pathlib import Path

# ---- Project root (same as in paths.txt) ----
PROJECT_ROOT = Path(r"C:\Users\HP EliteBook\Desktop\Liver Segmentation Project")

# ---- Folder we're working on right now ----
CATEGORY = "normal"   # <-- folder name inside CT_scans/ (change if yours is named differently)
CT_SCANS_DIR = PROJECT_ROOT / "CT_scans" / CATEGORY   # contains .nii.gz files directly

# ---- Where the RADAR repo lives (cloned in the next step if missing) ----
RADAR_REPO_DIR = PROJECT_ROOT / "damo-radar"
RADAR_INFERENCE_DIR = RADAR_REPO_DIR / "RADAR_inference"
CKPT_DIR = RADAR_REPO_DIR / "ckpt"

# ---- Where to save inference results (CSV) ----
RESULTS_DIR = PROJECT_ROOT / "RADAR_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SAVE_TAG = CATEGORY  # results will be saved as RADAR_infer_results_<CATEGORY>.csv (separate cache + CSV per category)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("CT_SCANS_DIR    :", CT_SCANS_DIR, "-> exists:", CT_SCANS_DIR.exists())
print("RADAR_REPO_DIR   :", RADAR_REPO_DIR, "-> exists:", RADAR_REPO_DIR.exists())
print("RESULTS_DIR      :", RESULTS_DIR)

# Safety net: if the folder name is wrong, show which folders actually exist
if not CT_SCANS_DIR.exists():
    print("\n[!] Folder not found. Available categories under CT_scans:")
    for p in sorted((PROJECT_ROOT / "CT_scans").iterdir()):
        if p.is_dir():
            print("   -", p.name)


PROJECT_ROOT   : C:\Users\HP EliteBook\Desktop\Liver Segmentation Project
CT_SCANS_DIR    : C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\CT_scans\normal -> exists: True
RADAR_REPO_DIR   : C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\damo-radar -> exists: True
RESULTS_DIR      : C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\RADAR_results


## 2) Download the repo (one-time)

If the repo isn't cloned yet, this cell clones it. If it's already there, it's skipped.


In [2]:
import subprocess

if not RADAR_REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/jhuanglabAI/damo-radar.git", str(RADAR_REPO_DIR)],
        check=True
    )
    print("Repo cloned.")
else:
    print("Repo already exists at:", RADAR_REPO_DIR)


Repo already exists at: C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\damo-radar


## 3) Install requirements (one-time)


In [3]:
import sys, subprocess

REQUIREMENTS_FILE = RADAR_REPO_DIR / "requirements.txt"

# Run this once (can take a while)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)], check=True)


CompletedProcess(args=['c:\\Users\\HP EliteBook\\anaconda\\python.exe', '-m', 'pip', 'install', '-r', 'C:\\Users\\HP EliteBook\\Desktop\\Liver Segmentation Project\\damo-radar\\requirements.txt'], returncode=0)

## 4) Download model weights (checkpoints) from HuggingFace



In [4]:
from huggingface_hub import snapshot_download

CKPT_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="radar-generalist/RADAR",
    repo_type="model",
    local_dir=str(CKPT_DIR),
    allow_patterns=[
        "checkpoint_radar_pretrain.pth",
        "bert-base-chinese/*",
        "infer_text_embedding_radar.pt",
        "**/infer_text_embedding_radar.pt",
    ],
)

TEXT_EMB_FILE = CKPT_DIR / "infer_text_embedding_radar.pt"

# Fall back to the auxiliary data repo if the file wasn't in the model repo
if not TEXT_EMB_FILE.exists():
    print("Not found in the model repo, checking RADAR-auxiliary-data ...")
    snapshot_download(
        repo_id="radar-generalist/RADAR-auxiliary-data",
        repo_type="dataset",
        local_dir=str(RADAR_REPO_DIR),
        allow_patterns=["**/infer_text_embedding_radar.pt", "infer_text_embedding_radar.pt"],
    )

print("\nckpt/ contents:")
for p in sorted(CKPT_DIR.rglob("*")):
    print(" -", p.relative_to(CKPT_DIR))

print("\ninfer_text_embedding_radar.pt found?", TEXT_EMB_FILE.exists())
if not TEXT_EMB_FILE.exists():
    print("⚠️ If it's still missing, we'll need to check the exact filename on the HuggingFace 'Files' tab manually.")


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]


ckpt/ contents:
 - .cache
 - .cache\huggingface
 - .cache\huggingface\.gitignore
 - .cache\huggingface\download
 - .cache\huggingface\download\bert-base-chinese
 - .cache\huggingface\download\bert-base-chinese\config.json.metadata
 - .cache\huggingface\download\bert-base-chinese\config_decoder.json.metadata
 - .cache\huggingface\download\bert-base-chinese\pytorch_model.bin.metadata
 - .cache\huggingface\download\bert-base-chinese\tokenizer.json.metadata
 - .cache\huggingface\download\bert-base-chinese\tokenizer_config.json.metadata
 - .cache\huggingface\download\bert-base-chinese\vocab.txt.metadata
 - .cache\huggingface\download\checkpoint_radar_pretrain.pth.metadata
 - bert-base-chinese
 - bert-base-chinese\config.json
 - bert-base-chinese\config_decoder.json
 - bert-base-chinese\pytorch_model.bin
 - bert-base-chinese\tokenizer.json
 - bert-base-chinese\tokenizer_config.json
 - bert-base-chinese\vocab.txt
 - checkpoint_radar_pretrain.pth
 - infer_text_embedding_merlin.pt
 - infer_tex

## 5) Check for a GPU

An NVIDIA GPU (CUDA) is strongly recommended, since RADAR's own code calls `model.cuda()` directly.


In [5]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected — the next cell will patch the code to run on CPU instead (much slower).")


CUDA available: False
⚠️ No GPU detected — the next cell will patch the code to run on CPU instead (much slower).


## 6) If there's no GPU — patch the code to run on CPU



In [6]:
import re
import shutil
import torch

FILES_TO_PATCH = [
    RADAR_INFERENCE_DIR / "inference_demo.py",
    RADAR_INFERENCE_DIR / "dynamic_network_architectures" / "architectures" / "resnet_vl.py",
]

DEVICE_SHIM = RADAR_INFERENCE_DIR / "_radar_device.py"
if not DEVICE_SHIM.exists():
    DEVICE_SHIM.write_text(
        'import torch\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\n',
        encoding="utf-8",
    )

MARKER = "# --- patched: auto CPU/GPU device ---"

for f in FILES_TO_PATCH:
    text = f.read_text(encoding="utf-8")
    changed = False

    backup = f.with_suffix(f.suffix + ".orig")
    if not backup.exists():
        shutil.copy(f, backup)
        print(f"{f.name}: backup saved -> {backup.name}")

    if MARKER not in text:
        text = text.replace(
            "import torch\n",
            "import torch\n" + MARKER + "\nfrom _radar_device import DEVICE as _DEVICE\n",
            1,
        )
        text = re.sub(r"\.cuda\(\)", ".to(_DEVICE)", text)
        text = text.replace(
            "torch.cuda.empty_cache()",
            "(torch.cuda.empty_cache() if torch.cuda.is_available() else None)",
        )
        changed = True
    else:
        print(f"{f.name}: device patch already applied.")

    # Fix: hardcoded relative path for the text embeddings file -> use model_root instead
    # (kept independent of the marker above so it still applies even if you ran an older
    # version of this cell before)
    buggy_line = "torch.load('../ckpt/infer_text_embedding_radar.pt')"
    fixed_line = "torch.load(os.path.join(model_root, 'infer_text_embedding_radar.pt'), map_location=_DEVICE)"
    if buggy_line in text:
        text = text.replace(buggy_line, fixed_line)
        changed = True
        print(f"{f.name}: fixed hardcoded text-embedding path.")

    # Fix: add map_location if an earlier version of the patch fixed the path but not this
    old_fixed_no_maplocation = "torch.load(os.path.join(model_root, 'infer_text_embedding_radar.pt'))"
    if old_fixed_no_maplocation in text:
        text = text.replace(old_fixed_no_maplocation, fixed_line)
        changed = True
        print(f"{f.name}: added map_location to the text-embedding load.")

    if changed:
        f.write_text(text, encoding="utf-8")
        print(f"{f.name}: saved.")
    else:
        print(f"{f.name}: nothing new to patch.")

print("\nDevice that will be used:", "cuda" if torch.cuda.is_available() else "cpu")


inference_demo.py: device patch already applied.
inference_demo.py: nothing new to patch.
resnet_vl.py: device patch already applied.
resnet_vl.py: nothing new to patch.

Device that will be used: cpu


## 7) Import RADAR's code

Important: `MODEL_ROOT` and `CONFIGS_ROOT` must be set **before** importing `inference_demo`,
since the module reads them once at import time and defaults to `../ckpt` (relative to `RADAR_inference/`).
We point them at the absolute `CKPT_DIR` path so this works from inside a notebook without changing the working directory.


In [7]:
import os, sys

os.environ["MODEL_ROOT"] = str(CKPT_DIR)
os.environ["CONFIGS_ROOT"] = str(CKPT_DIR)

if str(RADAR_INFERENCE_DIR) not in sys.path:
    sys.path.insert(0, str(RADAR_INFERENCE_DIR))

import importlib, inference_demo                      # noqa: E402
importlib.reload(inference_demo)                      # always use the latest edited file
from inference_demo import initialize, inference, assemble_csv, DataFolder  # noqa: E402

print("RADAR code imported successfully.")


c:\Users\HP EliteBook\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


RADAR code imported successfully.


## 8) Load the model into memory

This loads the vision encoder, text encoder and the pretrained weights, and moves them onto the GPU (or CPU, per the patch above).


In [8]:
init_returns = initialize()  # (pad_func, model)



--> Start initializing...
--> ckpt_path:  C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\damo-radar\ckpt\checkpoint_radar_pretrain.pth

--> Initialize done


## 8.5) (Optional) Quick smoke test on 2-3 cases

Especially if you're running on CPU, it's worth confirming everything works on a small sample
before committing to all 200 cases and waiting hours.


In [9]:
# import shutil

# N_TEST = 3
# TEST_DIR = PROJECT_ROOT / "RADAR_results" / "_smoke_test_input"
# TEST_DIR.mkdir(parents=True, exist_ok=True)

# sample_files = sorted(CT_SCANS_DIR.glob("*.nii.gz"))[:N_TEST]
# for f in sample_files:
#     dst = TEST_DIR / f.name
#     if not dst.exists():
#         shutil.copy(f, dst)

# print(f"Prepared {len(sample_files)} test case(s) in: {TEST_DIR}")

# # Run inference on this small sample only (different save_tag so it doesn't mix with the full run)
# inference(init_returns, str(TEST_DIR), str(RESULTS_DIR), "smoke_test")


In [10]:
# import pandas as pd

# smoke_csv = RESULTS_DIR / "RADAR_infer_results_smoke_test.csv"
# pd.read_csv(smoke_csv)


## 8.9) Pre-flight check: empty / corrupt files

Your last run crashed on `AC4215483.nii.gz` because it was a **0-byte file** (bad copy/download).
This cell lists such files. Re-copy them from the original source, or just leave them: the new code skips them and logs them to `RADAR_errors_<tag>.csv`.


In [11]:
DEEP_CHECK = False   # True = also test that every .nii.gz can be fully decompressed (a few minutes)

import gzip
bad = []
files = sorted(CT_SCANS_DIR.glob("*.nii*"))
for f in files:
    if f.stat().st_size == 0:
        bad.append((f.name, "empty (0 bytes)"))
    elif DEEP_CHECK and f.name.endswith(".gz"):
        try:
            with gzip.open(f, "rb") as g:
                while g.read(1 << 24):
                    pass
        except Exception as e:
            bad.append((f.name, f"corrupt gzip: {e}"))

print(f"{len(files)} files checked, {len(bad)} problem(s):")
for name, why in bad:
    print("  -", name, "->", why)


105 files checked, 0 problem(s):


## 9) Run inference on all (resumable)

* After **every** case a small JSON is saved in `RADAR_results/_cache_<tag>/` and the main CSV is refreshed.
* If it crashes / you stop it / the laptop restarts: **just run this cell again**. Finished cases are skipped.
* Each finished case prints a timing line (`windows` vs `crops`) so you can see where the time goes.
* A broken file or an out-of-memory case is logged to `RADAR_errors_<tag>.csv` and the run continues.


In [12]:
inference(init_returns, str(CT_SCANS_DIR), str(RESULTS_DIR), SAVE_TAG)



--> Start inference.
--> 0 case(s) already finished (cache: C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\RADAR_results\_cache_normal)
--> 105 case(s) left to run


Infer:   0%|          | 0/105 [10:17<?, ?it/s]

[OK] AC42137e0.nii.gz size=[99, 367, 392] windows=8 crop_passes=5 | windows 392s + crops 200s = 596s


Infer:   1%|          | 1/105 [14:57<17:51:05, 617.94s/it]

[OK] AC42137fc.nii.gz size=[96, 403, 467] windows=4 crop_passes=2 | windows 193s + crops 81s = 280s


...

[OK] AC4245250.nii.gz size=[96, 283, 384] windows=2 crop_passes=0 | windows 95s + crops 0s = 98s


Infer:  99%|█████████▉| 104/105 [8:24:41<04:14, 254.72s/it]

[OK] AC4245deb.nii.gz size=[96, 406, 422] windows=4 crop_passes=5 | windows 192s + crops 200s = 397s


Infer: 100%|██████████| 105/105 [8:24:44<00:00, 288.43s/it]

--> 104 case(s) in C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\RADAR_results\RADAR_infer_results_normal.csv
evaluate done, save result_csv to C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\RADAR_results\RADAR_infer_results_normal.csv.


### (Optional) Rebuild the CSV from the cache at any time
Useful if you stopped the run early and just want the CSV of what's finished so far.

In [13]:
print(assemble_csv(DataFolder(str(CT_SCANS_DIR)), str(RESULTS_DIR), SAVE_TAG))

C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\RADAR_results\RADAR_infer_results_normal.csv


## 10) Review the raw results



In [14]:
import pandas as pd

results_csv = RESULTS_DIR / f"RADAR_infer_results_{SAVE_TAG}.csv"
df = pd.read_csv(results_csv)

print("Number of cases:", len(df))
print("Number of findings:", df.shape[1] - 1)
df.head()


Number of cases: 104
Number of findings: 146


,file_name,主动脉_主动脉夹层 (Aorta_Aortic dissection),主动脉_主动脉瘤 (Aorta_Aortic aneurysm),主动脉_粥样硬化 (Aorta_Atherosclerosis),主动脉_钙化 (Aorta_Calcification),十二指肠_占位 (Duodenum_Mass),十二指肠_囊袋状突出影 (Duodenum_Saccular outpouching),十二指肠_憩室 (Duodenum_Diverticulum),十二指肠_梗阻 (Duodenum_Obstruction),十二指肠_溃疡 (Duodenum_Ulcer),...,膀胱_软组织密度影 (Bladder_Soft-tissue attenuation lesion),门静脉_增宽 (Portal vein_Dilatation),门静脉_栓塞 (Portal vein_Thrombosis),门静脉_高压 (Portal vein_Hypertension),食管_增粗迂曲血管影 (Esophagus_Dilated and tortuous tubular opacities),食管_管壁增厚 (Esophagus_Wall thickening),食管_裂孔疝 (Esophagus_Hiatal hernia),食管_静脉扩张迂曲 (Esophagus_Dilated and tortuous veins),食管_静脉曲张 (Esophagus_Varices),骶骨_骨炎 (Sacrum_Osteitis)
0,AC42137e0.nii.gz,0.037412,0.009297,0.009868,0.017501,0.016998,0.018077,0.017300,0.024679,0.046088,...,0.010100,0.030098,0.001142,0.005899,0.020792,0.022257,0.075234,0.019238,0.017534,0.248971
1,AC42137fc.nii.gz,0.302164,0.201907,0.304680,0.307993,0.064728,0.027330,0.026445,0.126734,0.083183,...,0.015079,0.170816,0.017997,0.084841,0.251352,0.203932,0.715601,0.246223,0.238929,0.924360
2,AC4213858.nii.gz,0.153330,0.081641,0.366889,0.478655,0.523892,0.804760,0.807983,0.143462,0.347063,...,0.025493,0.009859,0.006296,0.004576,0.157910,0.819430,0.985684,0.152290,0.144674,0.039854
3,AC4213f19.nii.gz,0.775801,0.674633,0.823165,0.847504,0.087100,0.030770,0.029713,0.035894,0.121699,...,0.010749,0.012649,0.002236,0.004262,0.040482,0.030153,0.238160,0.037940,0.035025,0.023227
4,AC4213f72.nii.gz,0.045891,0.019551,0.011277,0.016794,0.093247,0.092426,0.090356,0.220474,0.538041,...,0.002856,0.009865,0.001891,0.003310,0.017908,0.007303,0.122700,0.016553,0.015011,0.137254


## 11) Make the results readable


In [15]:
import re
import pandas as pd

def english_finding_name(column_name: str) -> str:
    """Extract the 'Organ_Finding' English text from a column header
    like '肝_脂肪肝 (Liver_Steatotic liver disease)'."""
    match = re.search(r"\(([^)]+)\)", column_name)
    return match.group(1) if match else column_name

finding_columns = [c for c in df.columns if c != "file_name"]
english_names = {c: english_finding_name(c) for c in finding_columns}

# Long format: one row per (case, finding), sorted by probability within each case
long_rows = []
for _, row in df.iterrows():
    for col in finding_columns:
        long_rows.append({
            "file_name": row["file_name"],
            "finding": english_names[col],
            "probability": row[col],
        })

long_df = pd.DataFrame(long_rows)
long_df = long_df.sort_values(["file_name", "probability"], ascending=[True, False])

readable_csv = RESULTS_DIR / f"RADAR_infer_results_{SAVE_TAG}_readable.csv"
long_df.to_csv(readable_csv, index=False)
print("Saved a readable, ranked version to:", readable_csv)

# Show the top findings for the first 3 cases as a quick sanity check
TOP_N = 10
for file_name, group in list(long_df.groupby("file_name"))[:3]:
    print(f"\n=== {Path(file_name).name} ===")
    print(group.head(TOP_N)[["finding", "probability"]].to_string(index=False))


Saved a readable, ranked version to: C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\RADAR_results\RADAR_infer_results_normal_readable.csv

=== AC42137e0.nii.gz ===
                      finding  probability
                 Rib_Fracture     0.973472
            Lung_Pneumothorax     0.807232
          Lung_Patchy opacity     0.807115
Lung_Hyperattenuating opacity     0.798602
                  Lung_Nodule     0.732949
        Lung_Pleural effusion     0.676946
                    Lung_Mass     0.621206
         Rib_Bone destruction     0.574330
               Rib_Metastasis     0.571241
      Lung_Pulmonary collapse     0.562791

=== AC42137fc.nii.gz ===
                        finding  probability
                Sacrum_Osteitis     0.924360
        Esophagus_Hiatal hernia     0.715601
Large bowel_Mural calcification     0.703613
                   Rib_Fracture     0.648828
      Large bowel_Appendicolith     0.533991
             Small bowel_Lipoma     0.342246
            

## 11.5) Liver-only view




In [16]:
ORGAN_FILTER = "Liver"  # change this to focus on a different organ, e.g. "Gallbladder", "Pancreas"

liver_columns = [c for c in finding_columns if english_names[c].startswith(ORGAN_FILTER + "_")]
print(f"Found {len(liver_columns)} '{ORGAN_FILTER}' findings.")

# --- Wide table: one row per case, one column per liver finding ---
liver_wide = df[["file_name"] + liver_columns].copy()
liver_wide = liver_wide.rename(columns=english_names)
liver_wide_csv = RESULTS_DIR / f"RADAR_infer_results_{SAVE_TAG}_liver_only.csv"
liver_wide.to_csv(liver_wide_csv, index=False)
print("Saved liver-only wide table to:", liver_wide_csv)

# --- Ranked list per case, liver findings only ---
liver_long = long_df[long_df["finding"].str.startswith(ORGAN_FILTER + "_")].copy()
liver_long_csv = RESULTS_DIR / f"RADAR_infer_results_{SAVE_TAG}_liver_only_ranked.csv"
liver_long.to_csv(liver_long_csv, index=False)
print("Saved liver-only ranked list to:", liver_long_csv)

for file_name, group in list(liver_long.groupby("file_name"))[:3]:
    print(f"\n=== {Path(file_name).name} ===")
    print(group[["finding", "probability"]].to_string(index=False))


Found 18 'Liver' findings.
Saved liver-only wide table to: C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\RADAR_results\RADAR_infer_results_normal_liver_only.csv
Saved liver-only ranked list to: C:\Users\HP EliteBook\Desktop\Liver Segmentation Project\RADAR_results\RADAR_infer_results_normal_liver_only_ranked.csv

=== AC42137e0.nii.gz ===
                                                 finding  probability
                           Liver_Steatotic liver disease     0.007989
                        Liver_Intrahepatic calcification     0.004353
                                        Liver_Hemangioma     0.002443
                            Liver_Hypoattenuating lesion     0.000819
                                  Liver_Periportal edema     0.000800
                                              Liver_Cyst     0.000462
                               Liver_Nodular enhancement     0.000450
                 Liver_Intrahepatic bile duct dilatation     0.000181
                   